# 02 Exploratory Data Analysis (Stage 3)

**Training set only.** The test set stays untouched until Stage 12 (decision A6).

- **Part A:** data quality and leakage checks
- **Part B:** what is linked to churn (charts)
- **Part C:** overlapping columns and business insights

Every result is a count or a rate. We never print single customers.

## Setup

In [ ]:
# Tools: pandas for tables, and our own package for settings and loading.
import pandas as pd

from churn.config import POSITIVE_LABEL, TARGET_COLUMN
from churn.data import load_train_data

PERCENT = 100  # turns a share (0.288) into a percentage (28.8)
UNKNOWN_LABEL = "Unknown"  # the word the data uses for hidden missing values

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Load the TRAINING set only (the shared loader also runs our 3 checks).
train_df = load_train_data()

# True/False per customer: did they churn? Makes rates easy (mean = share True).
is_churn = train_df[TARGET_COLUMN].eq(POSITIVE_LABEL)
overall_churn_pct = is_churn.mean() * PERCENT

print(f"Training rows: {len(train_df):,} | columns: {train_df.shape[1]}")
print(f"Overall churn rate: {overall_churn_pct:.1f}%")

In [ ]:
# Helper used by every check below, so the logic is written once.
# Give it any grouping (e.g. "price unknown: True/False") and it returns,
# per group: number of customers, churn rate %, and the gap from the overall rate.
def churn_rate_by(groups: pd.Series, churned: pd.Series) -> pd.DataFrame:
    """Customers, churn rate (%) and gap from overall churn, per group.

    Args:
        groups: The group of each customer (same rows as churned).
        churned: True if the customer churned.

    Returns:
        One row per group: customers, churn_pct, gap_vs_overall_pct.
    """
    # dropna=False keeps missing values as their own group.
    summary = churned.groupby(groups, dropna=False).agg(
        customers="size", churn_pct="mean"
    )
    summary["churn_pct"] = summary["churn_pct"].mul(PERCENT).round(1)
    # Positive gap = this group churns MORE than average (in % points).
    overall_pct = churned.mean() * PERCENT
    summary["gap_vs_overall_pct"] = (summary["churn_pct"] - overall_pct).round(1)
    return summary

## A1. Missing values

Questions:
1. Are the customers missing the 6 usage columns the **same** customers each time?
2. Do customers with missing values churn more or less than average?

In [ ]:
# The 6 usage columns that were missing together in Stage 2.
USAGE_COLUMNS = [
    "MonthlyRevenue",
    "MonthlyMinutes",
    "TotalRecurringCharge",
    "DirectorAssistedCalls",
    "OverageMinutes",
    "RoamingCalls",
]

# isna() gives True per empty cell. any(axis=1) = row missing AT LEAST one,
# all(axis=1) = row missing ALL six. If both counts match, it is the same rows.
usage_missing = train_df[USAGE_COLUMNS].isna()
missing_any_usage = usage_missing.any(axis=1)
missing_all_usage = usage_missing.all(axis=1)
print(f"Rows missing at least one usage column: {missing_any_usage.sum():,}")
print(f"Rows missing all six usage columns:     {missing_all_usage.sum():,}")

# Are customers with missing usage also missing the % change columns?
missing_perc_change = train_df["PercChangeMinutes"].isna()
both_missing = (missing_all_usage & missing_perc_change).sum()
print(f"Missing % change columns: {missing_perc_change.sum():,}")
print(f"...of which also missing all usage columns: {both_missing:,}")

In [ ]:
# Churn rate for customers WITH vs WITHOUT each kind of missing value.
# True = value missing. A big gap means "missing" itself carries information.
missing_flags = {
    "usage columns missing": missing_all_usage,
    "% change columns missing": missing_perc_change,
    "AgeHH1 missing": train_df["AgeHH1"].isna(),
    "ServiceArea missing": train_df["ServiceArea"].isna(),
}

# pd.concat stacks the small tables into one, labelled by check name.
pd.concat(
    {name: churn_rate_by(flag, is_churn) for name, flag in missing_flags.items()},
    names=["check", "value_missing"],
)

## A2. Hidden missing values ("Unknown" and 0)

Questions:
1. Do customers with "Unknown" (or age/income 0) churn differently?
2. Are age 0 and income 0 the same customers (one "no demographic data" group)?

In [ ]:
# True = the value is a hidden "unknown".
hidden_unknown_flags = {
    "HandsetPrice = Unknown": train_df["HandsetPrice"].eq(UNKNOWN_LABEL),
    "MaritalStatus = Unknown": train_df["MaritalStatus"].eq(UNKNOWN_LABEL),
    "Homeownership = Unknown": train_df["Homeownership"].eq(UNKNOWN_LABEL),
    "AgeHH1 = 0": train_df["AgeHH1"].eq(0),
    "IncomeGroup = 0": train_df["IncomeGroup"].eq(0),
}

pd.concat(
    {
        name: churn_rate_by(flag, is_churn)
        for name, flag in hidden_unknown_flags.items()
    },
    names=["check", "is_unknown"],
)

In [ ]:
# Crosstab = a count table of two True/False columns against each other.
# If most customers sit on the diagonal (True/True, False/False),
# age 0 and income 0 mostly happen together.
pd.crosstab(
    hidden_unknown_flags["AgeHH1 = 0"],
    hidden_unknown_flags["IncomeGroup = 0"],
    rownames=["AgeHH1 = 0"],
    colnames=["IncomeGroup = 0"],
)

## A3. Retention columns: leakage check (decision A2)

Question: do these columns give away the answer?

Warning signs of leakage:
- A group with churn close to **100%** (or 0%)
- A link much stronger than any normal behaviour column

A strong link alone does not prove leakage. The real question is **timing**: was the call made before or after the snapshot?

In [ ]:
# The 3 columns flagged in decision A2.
RETENTION_COLUMNS = [
    "MadeCallToRetentionTeam",
    "RetentionCalls",
    "RetentionOffersAccepted",
]

pd.concat(
    {column: churn_rate_by(train_df[column], is_churn) for column in RETENTION_COLUMNS},
    names=["column", "value"],
)

## A4. Odd values

Questions:
1. How many customers have impossible negative values?
2. What do the very large accounts (`UniqueSubs`, `ActiveSubs`) look like?

We only look here. Fixes are decided in Stage 4 (Cleaning).

In [ ]:
# Columns that should never be negative (money and days).
# PercChange columns are left out: a % change CAN be negative.
NON_NEGATIVE_COLUMNS = [
    "MonthlyRevenue",
    "TotalRecurringCharge",
    "CurrentEquipmentDays",
]

negative_counts = pd.Series(
    {column: (train_df[column] < 0).sum() for column in NON_NEGATIVE_COLUMNS},
    name="customers_with_negative_value",
)
negative_counts

In [ ]:
# Spread of subscriber counts. High percentiles show how rare big accounts are.
# 0.99 = 99% of customers are at or below this value.
SUBSCRIBER_PERCENTILES = [0.5, 0.9, 0.99, 0.999]
train_df[["UniqueSubs", "ActiveSubs"]].describe(percentiles=SUBSCRIBER_PERCENTILES).T

In [ ]:
# Do big accounts churn differently? And a logic check:
# active subscribers should never be more than unique (total) subscribers.
LARGE_ACCOUNT_SUBS = 5  # more than this many subscribers = "large account"

is_large_account = train_df["UniqueSubs"].gt(LARGE_ACCOUNT_SUBS)
active_above_unique = train_df["ActiveSubs"].gt(train_df["UniqueSubs"]).sum()
print(f"Customers with ActiveSubs > UniqueSubs: {active_above_unique:,}\n")

churn_rate_by(is_large_account.rename("large_account"), is_churn)